# Baseline Comparison: ER1 → ER4 + E1

Cross-experiment analysis of all reference baselines and the static LLM control.

**Questions this notebook answers:**
1. Does communication help at all? (ER2-ER4 vs ER1)
2. Which comm scheme gives the best success/token frontier? (Pareto plot)
3. How do engineered schemas compare to symbolic intent? (ER2 vs ER3)
4. Does event-triggering help bandwidth efficiency? (ER4 vs ER2)
5. Does the LLM controller justify its cost? (E1 vs ER2/ER4)

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent.parent
RENDEZVOUS_ROOT = REPO_ROOT / "rendezvous_comm"
sys.path.insert(0, str(RENDEZVOUS_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.storage import ExperimentStorage, load_cross_experiment
from src.metrics import compute_transfer_score, compute_ate, compute_delta_return_per_msg
from src.plotting import (
    plot_baseline_comparison, plot_metric_radar,
    plot_success_vs_tokens, save_figure, set_style,
)

set_style()

## 1. Load All Experiment Results

In [ ]:
EXP_IDS = ["er1", "er2", "er3", "er4", "e1"]

cross_df = load_cross_experiment(EXP_IDS)

if cross_df.empty:
    print("No completed runs found. Run individual experiment notebooks first.")
else:
    print(f"Loaded {len(cross_df)} total runs across {cross_df['experiment'].nunique()} experiments")
    display(cross_df.groupby("experiment").size().to_frame("runs"))

## 2. Core Comparison: Success Rate Across Baselines

In [ ]:
if not cross_df.empty:
    # Bar chart: M1 success rate per experiment
    fig = plot_baseline_comparison(cross_df, metric="M1_success_rate")
    save_figure(fig, str(RENDEZVOUS_ROOT / "results/baseline_success_comparison.png"))
    plt.show()

    # Same for all core metrics
    for metric in ["M2_avg_return", "M3_avg_steps", "M4_avg_collisions"]:
        fig = plot_baseline_comparison(cross_df, metric=metric)
        plt.show()

## 3. Success vs Communication Cost (Pareto Frontier)

The core plot: can comm methods achieve better success without excessive token cost?

In [ ]:
if not cross_df.empty:
    # Compute per-experiment mean metrics for the scatter
    exp_means = {}
    for exp_id in EXP_IDS:
        mask = cross_df["experiment"] == exp_id
        if mask.any():
            exp_means[exp_id] = {
                col: cross_df.loc[mask, col].mean()
                for col in cross_df.columns if col.startswith("M")
            }

    if exp_means:
        fig = plot_success_vs_tokens(exp_means)
        save_figure(fig, str(RENDEZVOUS_ROOT / "results/pareto_success_vs_tokens.png"))
        plt.show()

## 4. Radar Chart: Multi-Metric Profile

In [ ]:
if not cross_df.empty and exp_means:
    fig = plot_metric_radar(exp_means)
    save_figure(fig, str(RENDEZVOUS_ROOT / "results/radar_comparison.png"))
    plt.show()

## 5. Derived Metrics: ATE, Transfer Score, Delta Return/Msg

In [ ]:
if not cross_df.empty and "er1" in exp_means:
    er1_m = exp_means["er1"]
    print("=== Derived Metrics (relative to ER1 no-comm baseline) ===\n")

    for exp_id in ["er2", "er3", "er4", "e1"]:
        if exp_id not in exp_means:
            continue
        m = exp_means[exp_id]
        label = {"er2": "Eng. Schema", "er3": "Symbolic Intent",
                 "er4": "Event-Triggered", "e1": "Static LLM"}[exp_id]

        ate = compute_ate(m, er1_m)  # comm vs no-comm
        delta_per_msg = compute_delta_return_per_msg(m, er1_m)
        transfer = compute_transfer_score(er1_m, m)

        print(f"--- {exp_id.upper()}: {label} ---")
        print(f"  ATE (success):           {ate['ate_success']:+.4f}")
        print(f"  ATE (return):            {ate['ate_return']:+.4f}")
        print(f"  ΔReturn/msg:             {delta_per_msg:+.6f}")
        print(f"  Transfer score:          {transfer:.4f}")
        print()

## 6. Summary Table for Paper

In [ ]:
if not cross_df.empty:
    summary = cross_df.groupby("experiment").agg({
        "M1_success_rate": ["mean", "std"],
        "M2_avg_return": ["mean", "std"],
        "M3_avg_steps": ["mean", "std"],
        "M4_avg_collisions": ["mean", "std"],
        "M5_avg_tokens": ["mean", "std"],
    }).round(4)

    # Flatten column multi-index for cleaner display
    summary.columns = [f"{col[0]}_{col[1]}" for col in summary.columns]
    display(summary)

    # Export to LaTeX for paper
    latex_path = RENDEZVOUS_ROOT / "results/baseline_summary_table.tex"
    summary.to_latex(latex_path, float_format="%.3f")
    print(f"\nLaTeX table saved to: {latex_path}")